# TP 2 — Clustering : k-means, DBSCAN, agglomératif

**Objectifs**

- Comparer trois algorithmes de clustering sur des jeux synthétiques 2D.
- Choisir le nombre de clusters par la méthode du coude et le coefficient de silhouette.
- Identifier les cas où chaque algorithme excelle ou échoue.

**Durée indicative : 50 minutes.**

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from sklearn.datasets import make_blobs, make_circles, make_moons
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)

# Trois jeux de données 2D
X_blobs, y_blobs = make_blobs(n_samples=400, centers=4, cluster_std=0.7, random_state=0)
X_moons, y_moons = make_moons(n_samples=400, noise=0.08, random_state=0)
X_circles, y_circles = make_circles(n_samples=400, factor=0.5, noise=0.06, random_state=0)

## Exercice 1 — Choisir k pour k-means via coude et silhouette

Sur `X_blobs` (standardisé) :

1. Pour `k` de 2 à 10, calculez l'inertie de `KMeans(n_clusters=k, n_init=10, random_state=0)`.
2. Pour la même plage, calculez la silhouette globale.
3. Tracez les deux courbes côte à côte. Lequel des deux critères suggère le meilleur k ? Est-ce cohérent avec le `centers=4` utilisé pour générer les données ?

<details>
<summary>💡 Coup de pouce — choisir k pour k-means</summary>

**🎯 But :** déterminer automatiquement le nombre de clusters à demander à k-means, qui ne le devine pas.

**Méthode du coude — inertie**

```python
from sklearn.cluster import KMeans
inerties = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    inerties.append(km.inertia_)
```

`km.inertia_` = somme des **distances² de chaque point à son centroïde**. Elle décroit toujours avec `k` (plus de clusters = points plus proches du leur).

**Tracer et identifier le coude**

```python
plt.plot(range(1, 11), inerties, marker='o')
plt.xlabel('k'); plt.ylabel('Inertie')
```

Le « coude » = l'endroit où la décroissance ralentit franchement. Avant le coude, ajouter un cluster réduit beaucoup l'inertie ; après, peu. C'est subjectif à l'œil.

**Méthode du silhouette — plus objective**

```python
from sklearn.metrics import silhouette_score
sils = []
for k in range(2, 11):                  # silhouette pas définie pour k=1
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    sils.append(silhouette_score(X, km.labels_))
```

Le **score silhouette** est dans [-1, 1] :
- Proche de **1** : clusters bien séparés, points proches de leur centroïde et loin des autres.
- Proche de **0** : chevauchement.
- Négatif : un point est mieux dans un autre cluster que le sien → mauvais clustering.

**Choisir k**

```python
k_opt = 2 + np.argmax(sils)            # +2 car on a commencé à k=2
```

**Pourquoi `n_init=10` ?**

k-means est sensible à l'initialisation (minima locaux). `n_init=10` répète 10 fois avec des inits différentes et garde la meilleure inertie. **Toujours mettre `n_init=10`** dans des cours/TPs pour des résultats reproductibles.

</details>

In [2]:
# TODO


## Exercice 2 — Comparer 3 algorithmes sur `make_moons`

Sur `X_moons` (standardisé) :

1. Appliquez `KMeans(n_clusters=2, n_init=10, random_state=0)`.
2. Appliquez `AgglomerativeClustering(n_clusters=2, linkage="ward")`.
3. Appliquez `DBSCAN(eps=0.3, min_samples=5)`.
4. Tracez les 3 résultats côte à côte. Calculez l'ARI (`adjusted_rand_score`) par rapport à `y_moons` pour chacun.

Discussion : quel algorithme retrouve les deux croissants ? Pourquoi k-means échoue-t-il ?

<details>
<summary>💡 Coup de pouce — comparer k-means, agglo, DBSCAN sur `make_moons`</summary>

**🎯 But :** comprendre pourquoi un algorithme de clustering échoue sur certaines formes alors qu'un autre réussit.

**Le dataset**

```python
from sklearn.datasets import make_moons
X, y_true = make_moons(n_samples=300, noise=0.1, random_state=0)
```

Deux **croissants entrelacés**. Forme non convexe → un cluster ne tient pas dans une boule.

**Comparer 3 algos avec ARI**

```python
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import adjusted_rand_score

models = {
    'kmeans': KMeans(n_clusters=2, n_init=10, random_state=0),
    'ward':   AgglomerativeClustering(n_clusters=2),
    'dbscan': DBSCAN(eps=0.2, min_samples=5),
}
for name, model in models.items():
    labels = model.fit_predict(X)
    ari = adjusted_rand_score(y_true, labels)
    print(f"{name:8s} ARI = {ari:.3f}")
```

**ARI (Adjusted Rand Index)**
- 1.0 = clustering parfait
- 0.0 = aléatoire
- Négatif = pire qu'aléatoire

C'est la métrique standard quand on a les vrais labels (et qu'on veut comparer des étiquetages malgré la **permutation libre des numéros de clusters**).

**Ce que vous devriez voir**

| Algorithme | ARI sur `moons` | Pourquoi |
|---|---|---|
| KMeans | ≈ 0.2 | Cherche des clusters convexes → coupe les croissants en deux verticalement |
| Ward (agglo) | ≈ 0.4 | Mieux mais limité par la métrique euclidienne |
| **DBSCAN** | **≈ 1.0** | Suit la densité locale → retrouve les croissants exactement |

**Visualiser les 3 résultats côte à côte**

```python
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (name, model) in zip(axes, models.items()):
    labels = model.fit_predict(X)
    ax.scatter(X[:, 0], X[:, 1], c=labels, cmap='tab10', s=10)
    ax.set_title(f"{name}
ARI={adjusted_rand_score(y_true, labels):.2f}")
```

</details>

In [3]:
# TODO


## Exercice 3 — Cas des cercles concentriques

Refaites la même comparaison sur `X_circles`. Quel(s) algorithme(s) résolvent ce cas, et pourquoi ?

Bonus : appliquez `DBSCAN` avec différentes valeurs de `eps` (0.1, 0.2, 0.4) et observez l'impact sur le nombre de clusters détectés et le nombre de points marqués comme bruit (label −1).

<details>
<summary>💡 Coup de pouce — cercles concentriques + tuning DBSCAN</summary>

**🎯 But :** comprendre que les algos basés sur la distance euclidienne (k-means, Ward) échouent sur des structures imbriquées, et apprendre à tuner DBSCAN.

**Le dataset**

```python
from sklearn.datasets import make_circles
X, y_true = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=0)
```

Deux cercles concentriques. **Aucune ligne droite ne les sépare** → encore plus dur que les croissants.

**Pourquoi k-means échoue**

k-means cherche des **centroïdes** (centres de masse). Sur 2 cercles concentriques, les deux centres de masse coïncident au centre commun. K-means coupe l'image en quartiers, pas en cercles.

**Pourquoi Ward échoue aussi**

Ward agglomère les points qui minimisent la **variance intra-cluster**. Même piège : la métrique euclidienne ne distingue pas « être sur le cercle intérieur » de « être sur le cercle extérieur ».

**Pourquoi DBSCAN réussit**

DBSCAN ne cherche pas un centre. Il regarde la **densité locale** : un point fait partie d'un cluster si beaucoup d'autres sont à `eps` autour. Les deux cercles sont denses chacun de leur côté → ils ressortent comme deux clusters distincts.

**Tuner `eps` (Bonus)**

```python
for eps in [0.05, 0.1, 0.15, 0.2, 0.3, 0.5]:
    labels = DBSCAN(eps=eps, min_samples=5).fit_predict(X)
    n_clusters = len(set(labels) - {-1})        # -1 = points bruit
    n_noise = (labels == -1).sum()
    print(f"eps={eps:.2f} → {n_clusters} clusters, {n_noise} points bruit")
```

Vous observerez :
- `eps` **trop petit** (0.05) : tout est marqué bruit (`-1`).
- `eps` **juste** (~0.15) : 2 clusters propres.
- `eps` **trop grand** (0.5) : un seul gros cluster (les deux cercles fusionnent).

**Heuristique pour `eps`**

Le **k-distance plot** : tracer la distance au k-ème voisin trié → coude indique le bon `eps`.

```python
from sklearn.neighbors import NearestNeighbors
nn = NearestNeighbors(n_neighbors=5).fit(X)
dists, _ = nn.kneighbors(X)
plt.plot(np.sort(dists[:, -1]))
```

Le coude donne une estimation de `eps`.

</details>

In [4]:
# TODO
